# Sahayak - YOLO11n vs YOLO26n vs YOLO26s study (Colab)
**Before you start:** Runtime -> Change runtime type -> **T4 GPU** -> Save. Then run the cells **top to bottom**, one at a time.
Results are saved to your Google Drive (`MyDrive/sahayak_lab`), so a disconnect does not lose them. Keep this tab open.

Rough time (estimate, 20 epochs): dataset 20-40 min, each model 30-60 min.

In [ ]:
!nvidia-smi | head -15

## 1. Connect Google Drive (a popup asks for permission)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
for d in ('lab_out', 'lab_runs'):
    os.makedirs(f'/content/drive/MyDrive/sahayak_lab/{d}', exist_ok=True)

## 2. Get the project and install packages

In [ ]:
!rm -rf /content/sahayak
!git clone -q https://github.com/shreyaaaah/fewshot-object-detection-for-visually-impaired-person.git /content/sahayak
%cd /content/sahayak
!ln -sfn /content/drive/MyDrive/sahayak_lab/lab_out lab_out
!ln -sfn /content/drive/MyDrive/sahayak_lab/lab_runs lab_runs
!pip -q install -U ultralytics fiftyone pandas matplotlib

## 3. Build the dataset (28 COCO classes)
**Check the printout:** about 8000 train / 1000 val / 1000 test, and no class listed under 'WARNING: fewer than 50 training boxes'. If something looks wrong, stop and tell me.

In [ ]:
!python eval/make_dataset.py

## 4. Baseline: the pretrained models, no fine-tuning (about 5 min)

In [ ]:
DATA = 'datasets/sahayak28/data.yaml'
EPOCHS = 20
!python lab.py eval --data {DATA} --weights yolo11n.pt yolo26n.pt yolo26s.pt --imgsz 640
!python lab.py speed --imgsz 640

## 5. Fine-tune YOLO26n (your main model) - run this first

In [ ]:
!python lab.py train --data {DATA} --model yolo26n.pt --tag ft_yolo26n --epochs {EPOCHS}

## 6. Fine-tune YOLO11n (for the comparison)

In [ ]:
!python lab.py train --data {DATA} --model yolo11n.pt --tag ft_yolo11n --epochs {EPOCHS}

## 7. Fine-tune YOLO26s (optional - skip if you are short on time)

In [ ]:
!python lab.py train --data {DATA} --model yolo26s.pt --tag ft_yolo26s --epochs {EPOCHS}

## 8. Optional transfer-learning ablation: YOLO26n with the backbone frozen

In [ ]:
!python lab.py train --data {DATA} --model yolo26n.pt --tag ft_yolo26n_head --epochs {EPOCHS} --freeze 10

## 9. Make the report, plots and tables

In [ ]:
!python lab.py report
from IPython.display import Image, display, Markdown
display(Markdown(open('lab_out/REPORT.md').read()))
for f in ['fig3_before_after', 'fig1_loss_curves', 'fig4_pr_f1_curves', 'fig5_confusion', 'fig6_per_class_ap50', 'fig7_speed_vs_accuracy']:
    display(Image(f'lab_out/{f}.png'))

## 10. Download everything

In [ ]:
!cd /content/sahayak && zip -qr results.zip lab_out
!cp lab_runs/ft_yolo26n/weights/best.pt /content/best_yolo26n.pt
from google.colab import files
files.download('/content/sahayak/results.zip')
files.download('/content/best_yolo26n.pt')